In [16]:
import pandas as pd
import re

# Charger le df concaténé des deux législatures
df = pd.read_csv(
    "../data/interim/extract_15_16_concat.csv",
    low_memory=False,
    dtype={
        "id_orateur": str  # éviter identification en float avant d'avoir ajouté le "PA"
    },
)

print("Shape du df chargé : ", df.shape)


Shape du df chargé :  (1127838, 29)


In [ ]:
# Je supose que c'est un truc du genre la date :
df["dateseance_ts"] = pd.to_datetime(df["dateSeance"], format="%Y%m%d%H%M%S%f")
df["dateSeance_day"] = df["dateseance_ts"].dt.normalize()

# Filtrer les séances Lamartine à exclure du df

# Pour pouvoir utiliser la colonne numSeanceJour, nécessaire de changer la modalité unique en variable numérique
df["numSeanceJour"] = df["numSeanceJour"].replace("Unique", "0")

# Puis de changer les numéros en vraies variables numériques
df["numSeanceJour"] = pd.to_numeric(df["numSeanceJour"], errors="raise")

# Liste des critères de filtrage
filtres = [
    {
        "dateSeance_day": "2024-02-26",
        "numSeanceJour": 1,
        "valeur_ptsodj": 1,
    },  # « L’école publique face aux politiques de tri social »
    {
        "dateSeance_day": "2024-01-19",
        "numSeanceJour": 0,
        "valeur_ptsodj": 2,
    },  # « Essais nucléaires en Polynésie française : indemnisation des victimes directes, indirectes et transgénérationnelles et réparations environnementales ».
    {
        "dateSeance_day": "2024-01-17",
        "numSeanceJour": 2,
        "valeur_ptsodj": 1,
    },  # le sans-abrisme est-il le réceptacle des échecs des politiques publiques ?
    {
        "dateSeance_day": "2023-11-27",
        "numSeanceJour": 0,
        "valeur_ptsodj": 2,
    },  # « Le chlordécone en Martinique et en Guadeloupe, l’action de l’État face aux nécessaires réparations. »
    {
        "dateSeance_day": "2023-05-05",
        "numSeanceJour": 2,
        "valeur_ptsodj": 2,
    },  # « Quelles réponses à l’envolée des prix des produits de grande consommation ? ».
    {
        "dateSeance_day": "2023-04-03",
        "numSeanceJour": 1,
        "valeur_ptsodj": 2,
    },  # lutte contre le terrorisme d’extrême droite
    {
        "dateSeance_day": "2023-04-03",
        "numSeanceJour": 2,
        "valeur_ptsodj": 1,
    },  # « Pour une politique ambitieuse du grand âge ».
    {
        "dateSeance_day": "2023-04-03",
        "numSeanceJour": 2,
        "valeur_ptsodj": 3,
    },  # « L’école inclusive, une réalité ? » #TODO:bug ou je trouve pas ? -> séance jour 1
    {
        "dateSeance_day": "2023-04-03",
        "numSeanceJour": 2,
        "valeur_ptsodj": 1,
    },  # « Pour une politique ambitieuse du grand âge »
    {"dateSeance_day": "2023-01-31", "numSeanceJour": 1, "valeur_ptsodj": 1},  #
    {
        "dateSeance_day": "2023-02-27",
        "numSeanceJour": 1,
        "valeur_ptsodj": 1,
    },  # réforme des retraites et la pénibilité
    {
        "dateSeance_day": "2023-02-27",
        "numSeanceJour": 1,
        "valeur_ptsodj": 2,
    },  # débat sur les retraites et la protection sociale dans la fonction publique
    {
        "dateSeance_day": "2024-02-26",
        "numSeanceJour": 2,
        "valeur_ptsodj": 2,
    },  #  « Décentralisation des politiques publiques agricoles : simplifier, adapter et mieux associer les territoires ».
    {
        "dateSeance_day": "2021-03-22",
        "numSeanceJour": 1,
        "valeur_ptsodj": 2,
    },  # débat sur la dimension logistique de la stratégie vaccinale contre l’épidémie de covid-19
    {
        "dateSeance_day": "2024-05-06",
        "numSeanceJour": 1,
        "valeur_ptsodj": 1,
    },  # « Bilan des politiques publiques de défense et de promotion de la laïcité ».
    {
        "dateSeance_day": "2024-04-05",
        "numSeanceJour": 0,
        "valeur_ptsodj": 2,
    },  # « Quel grand plan pour l’emploi des seniors, après la réforme des retraites et celle de l’assurance chômage ? »
    {
        "dateSeance_day": "2024-04-05",
        "numSeanceJour": 0,
        "valeur_ptsodj": 1,
    },  #  « La place dans la société et dans le droit des familles monoparentales »
    {
        "dateSeance_day": "2024-04-03",
        "numSeanceJour": 2,
        "valeur_ptsodj": 1,
    },  # Bilan des réformes de l’assurance chômage depuis 2017
    {
        "dateSeance_day": "2024-04-03",
        "numSeanceJour": 1,
        "valeur_ptsodj": 3,
    },  # Conditions d’accueil des enfants placés à l’aide sociale à l’enfance
    {
        "dateSeance_day": "2024-04-03",
        "numSeanceJour": 1,
        "valeur_ptsodj": 2,
    },  # Défaillances de l’aide sociale à l’enfance
    {
        "dateSeance_day": "2024-02-28",
        "numSeanceJour": 2,
        "valeur_ptsodj": 1,
    },  # Conséquences de la loi « immigration » sur les enfants étrangers placés à l’aide sociale à l’enfance
    {
        "dateSeance_day": "2024-02-28",
        "numSeanceJour": 1,
        "valeur_ptsodj": 1,
    },  # Défaillances de l’aide sociale à l’enfance
    {
        "dateSeance_day": "2020-01-09",
        "numSeanceJour": 0,
        "valeur_ptsodj": 2,
    },  # projet hercule
    {
        "dateSeance_day": "2023-01-09",
        "numSeanceJour": 1,
        "valeur_ptsodj": 2,
    },  # Réforme de la voie professionnelle
    {
        "dateSeance_day": "2022-01-31",
        "numSeanceJour": 1,
        "valeur_ptsodj": 2,
    },  # Évaluation du plan gouvernemental L’État plus fort en Seine-Saint-Denis
    {
        "dateSeance_day": "2022-01-06",
        "numSeanceJour": 1,
        "valeur_ptsodj": 2,
    },  # Sahara occidental
    {
        "dateSeance_day": "2022-01-06",
        "numSeanceJour": 1,
        "valeur_ptsodj": 1,
    },  # Légalisation du cannabis : évolutions européennes,
    {
        "dateSeance_day": "2021-05-03",
        "numSeanceJour": 1,
        "valeur_ptsodj": 3,
    },  # Bilan de la loi ÉGALIM sur la rémunération des agriculteurs
    {
        "dateSeance_day": "2020-02-06",
        "numSeanceJour": 0,
        "valeur_ptsodj": 2,
    },  # D ébat sur les allégements de la fiscalité
]

# Initialiser une liste pour stocker les DataFrames filtrés
dfs_filtres = []

# Appliquer les filtres
for filtre in filtres:
    date = filtre["dateSeance_day"]
    num_seance = filtre["numSeanceJour"]
    valeur_pt = filtre[
        "valeur_ptsodj"
    ]  # TODO: LM : doute ici avec structure des fichiers

    # Filtrer par date et numéro de séance
    df_temp = df[(df["dateSeance_day"] == date) & (df["numSeanceJour"] == num_seance)]

    # Si une valeur de point est spécifiée, filtrer aussi par cette valeur
    if valeur_pt is not None:
        df_temp = df_temp[df_temp["valeur_ptsodj"] == valeur_pt]

    # Ajouter le DataFrame filtré à la liste
    dfs_filtres.append(df_temp)

# Concaténer tous les DataFrames filtrés
df_filtres = pd.concat(dfs_filtres, ignore_index=True)

In [18]:
df_filtres

,uid,SeanceRef,SessionRef,dateSeance,dateSeanceJour,numSeanceJour,numSeance,typeAssemblee,legislature,session,...,code_parole,id_syceron,roledebat,nom_orateur,qualite_orateur,id_orateur,stime,texte,dateseance_ts,dateSeance_day
0,CRSANR5L16S2023O1N201,RUANR5L16S2023IDS26963,SCR5A2023O1,20230403213000000,lundi 03 avril 2023,2,201,AN,16,Session ordinaire 2022-2023,...,NaN,3077628,president,Mme la présidente,NaN,720286,938.90,L’ordre du jour appelle le débat sur le thème ...,2023-04-03 21:30:00,2023-04-03
1,CRSANR5L16S2023O1N201,RUANR5L16S2023IDS26963,SCR5A2023O1,20230403213000000,lundi 03 avril 2023,2,201,AN,16,Session ordinaire 2022-2023,...,PAROLE_1_2,3077632,NaN,"Mme Myriam El Khomri,","ancienne ministre du travail, auteure d’un rap...",-125309,973.02,"Je suis accompagnée par Mme Dafna Mouchenik, a...",2023-04-03 21:30:00,2023-04-03
2,CRSANR5L16S2023O1N201,RUANR5L16S2023IDS26963,SCR5A2023O1,20230403213000000,lundi 03 avril 2023,2,201,AN,16,Session ordinaire 2022-2023,...,PAROLE_1_1,3077636,NaN,Mme la présidente,NaN,720286,1160.27,La parole est à Mme Dafna Mouchenik.,2023-04-03 21:30:00,2023-04-03
3,CRSANR5L16S2023O1N201,RUANR5L16S2023IDS26963,SCR5A2023O1,20230403213000000,lundi 03 avril 2023,2,201,AN,16,Session ordinaire 2022-2023,...,PAROLE_1_2,3077637,NaN,Mme Dafna Mouchenik,NaN,-125309,1206.30,Je suis directrice d’un service d’aide à domic...,2023-04-03 21:30:00,2023-04-03
4,CRSANR5L16S2023O1N201,RUANR5L16S2023IDS26963,SCR5A2023O1,20230403213000000,lundi 03 avril 2023,2,201,AN,16,Session ordinaire 2022-2023,...,PAROLE_1_1,3077639,NaN,Mme la présidente,NaN,720286,1314.51,"La parole est à M. Thierry d’Aboville, secréta...",2023-04-03 21:30:00,2023-04-03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2993,CRSANR5L15S2020O1N137,NaN,NaN,20200206090000000,jeudi 06 février 2020,0,137,AN,15,session ordinaire 2019-2020,...,NaN,2003290,NaN,Mme Émilie Cariou,NaN,720298,NaN,Ah non ! C’est quand même un flux financier qu...,2020-02-06 09:00:00,2020-02-06
2994,CRSANR5L15S2020O1N137,NaN,NaN,20200206090000000,jeudi 06 février 2020,0,137,AN,15,session ordinaire 2019-2020,...,PAROLE_1_2,2003291,NaN,Mme Agnès Pannier-Runacher,secrétaire d’État,759832,NaN,"Ce n’est pas ce que je dis ! Simplement, les p...",2020-02-06 09:00:00,2020-02-06
2995,CRSANR5L15S2020O1N137,NaN,NaN,20200206090000000,jeudi 06 février 2020,0,137,AN,15,session ordinaire 2019-2020,...,NaN,2003292,NaN,Mme Émilie Cariou,NaN,720298,NaN,On le sait bien !,2020-02-06 09:00:00,2020-02-06
2996,CRSANR5L15S2020O1N137,NaN,NaN,20200206090000000,jeudi 06 février 2020,0,137,AN,15,session ordinaire 2019-2020,...,PAROLE_1_2,2003293,NaN,Mme Agnès Pannier-Runacher,secrétaire d’État,759832,NaN,"Lorsqu’on produit en Chine, on sert beaucoup l...",2020-02-06 09:00:00,2020-02-06


In [19]:
df_filtres["id_acteur"].nunique()

306

In [20]:
df_filtres["id_orateur"].nunique()

285

In [21]:
df_filtres["nom_orateur"].nunique()

392

In [22]:
df_filtres["id_acteur"].isna().sum()

34

In [23]:
df_filtres["id_orateur"].isna().sum()

310

In [24]:
df_filtres["nom_orateur"].isna().sum()

45

In [25]:
test = df_filtres[["id_acteur", "id_orateur", "nom_orateur"]]
test

,id_acteur,id_orateur,nom_orateur
0,PA720286,720286,Mme la présidente
1,PA-125309,-125309,"Mme Myriam El Khomri,"
2,PA720286,720286,Mme la présidente
3,PA-125309,-125309,Mme Dafna Mouchenik
4,PA720286,720286,Mme la présidente
...,...,...,...
2993,PA720298,720298,Mme Émilie Cariou
2994,PA759832,759832,Mme Agnès Pannier-Runacher
2995,PA720298,720298,Mme Émilie Cariou
2996,PA759832,759832,Mme Agnès Pannier-Runacher


In [26]:
test

,id_acteur,id_orateur,nom_orateur
0,PA720286,720286,Mme la présidente
1,PA-125309,-125309,"Mme Myriam El Khomri,"
2,PA720286,720286,Mme la présidente
3,PA-125309,-125309,Mme Dafna Mouchenik
4,PA720286,720286,Mme la présidente
...,...,...,...
2993,PA720298,720298,Mme Émilie Cariou
2994,PA759832,759832,Mme Agnès Pannier-Runacher
2995,PA720298,720298,Mme Émilie Cariou
2996,PA759832,759832,Mme Agnès Pannier-Runacher


In [27]:
df_filtres["uid"].unique().tolist()

['CRSANR5L16S2023O1N201',
 'CRSANR5L16S2024O1N124',
 'CRSANR5L16S2024O1N098',
 'CRSANR5L16S2024O1N095',
 'CRSANR5L16S2024O1N063',
 'CRSANR5L16S2023O1N226',
 'CRSANR5L16S2023O1N128',
 'CRSANR5L16S2023O1N200',
 'CRSANR5L16S2023O1N156',
 'CRSANR5L16S2024O1N125',
 'CRSANR5L15S2021O1N207',
 'CRSANR5L16S2024O1N184',
 'CRSANR5L16S2024O1N171',
 'CRSANR5L16S2024O1N167',
 'CRSANR5L16S2024O1N166',
 'CRSANR5L16S2024O1N130',
 'CRSANR5L16S2024O1N129',
 'CRSANR5L15S2020O1N119',
 'CRSANR5L16S2023O1N106',
 'CRSANR5L15S2022O1N139',
 'CRSANR5L15S2022O1N112',
 'CRSANR5L15S2021O1N258',
 'CRSANR5L15S2020O1N137']

In [36]:
df_filtres[df_filtres["dateSeance_day"] == "2023-03-03"]

,uid,SeanceRef,SessionRef,dateSeance,dateSeanceJour,numSeanceJour,numSeance,typeAssemblee,legislature,session,...,code_parole,id_syceron,roledebat,nom_orateur,qualite_orateur,id_orateur,stime,texte,dateseance_ts,dateSeance_day


In [ ]:
# Je supose que c'est un truc du genre la date :
df["dateseance_ts"] = pd.to_datetime(df["dateSeance"], format="%Y%m%d%H%M%S%f")
df["dateSeance_day"] = df["dateseance_ts"].dt.normalize()

# Filtrer les séances Lamartine à exclure du df

# Pour pouvoir utiliser la colonne numSeanceJour, nécessaire de changer la modalité unique en variable numérique
df["numSeanceJour"] = df["numSeanceJour"].replace("Unique", "0")

# Puis de changer les numéros en vraies variables numériques
df["numSeanceJour"] = pd.to_numeric(df["numSeanceJour"], errors="raise")

# Liste des critères de filtrage
filtres = [
    {
        "dateSeance_day": "2023-04-03",
        "numSeanceJour": 2,
        "valeur_ptsodj": 1,
    },  # « Pour une politique ambitieuse du grand âge »
    {
        "dateSeance_day": "2024-02-26",
        "numSeanceJour": 1,
        "valeur_ptsodj": 1,
    },  # « L’école publique face aux politiques de tri social »
    {
        "dateSeance_day": "2024-01-19",
        "numSeanceJour": 0,
        "valeur_ptsodj": 2,
    },  # « Essais nucléaires en Polynésie française : indemnisation des victimes directes, indirectes et transgénérationnelles et réparations environnementales ».
    {
        "dateSeance_day": "2024-01-17",
        "numSeanceJour": 2,
        "valeur_ptsodj": 1,
    },  # le sans-abrisme est-il le réceptacle des échecs des politiques publiques ?
    {
        "dateSeance_day": "2023-11-27",
        "numSeanceJour": 0,
        "valeur_ptsodj": 2,
    },  # « Le chlordécone en Martinique et en Guadeloupe, l’action de l’État face aux nécessaires réparations. »
    {
        "dateSeance_day": "2023-05-05",
        "numSeanceJour": 2,
        "valeur_ptsodj": 2,
    },  # « Quelles réponses à l’envolée des prix des produits de grande consommation ? ».
    {
        "dateSeance_day": "2023-04-03",
        "numSeanceJour": 2,
        "valeur_ptsodj": 1,
    },  # « Pour une politique ambitieuse du grand âge ».
    {"dateSeance_day": "2023-01-31", "numSeanceJour": 1, "valeur_ptsodj": 1},  #
    {
        "dateSeance_day": "2023-04-03",
        "numSeanceJour": 1,
        "valeur_ptsodj": 3,
    },  # « L’école inclusive, une réalité ? »
    {
        "dateSeance_day": "2023-04-03",
        "numSeanceJour": 1,
        "valeur_ptsodj": 2,
    },  # lutte contre le terrorisme d’extrême droite
    {
        "dateSeance_day": "2023-02-27",
        "numSeanceJour": 1,
        "valeur_ptsodj": 1,
    },  # réforme des retraites et la pénibilité
    {
        "dateSeance_day": "2023-02-27",
        "numSeanceJour": 1,
        "valeur_ptsodj": 2,
    },  # débat sur les retraites et la protection sociale dans la fonction publique
    {
        "dateSeance_day": "2024-02-26",
        "numSeanceJour": 2,
        "valeur_ptsodj": 2,
    },  #  « Décentralisation des politiques publiques agricoles : simplifier, adapter et mieux associer les territoires ».
    {
        "dateSeance_day": "2021-03-22",
        "numSeanceJour": 1,
        "valeur_ptsodj": 2,
    },  # débat sur la dimension logistique de la stratégie vaccinale contre l’épidémie de covid-19
    {
        "dateSeance_day": "2024-05-06",
        "numSeanceJour": 1,
        "valeur_ptsodj": 1,
    },  # « Bilan des politiques publiques de défense et de promotion de la laïcité ».
    {
        "dateSeance_day": "2024-04-05",
        "numSeanceJour": 0,
        "valeur_ptsodj": 2,
    },  # « Quel grand plan pour l’emploi des seniors, après la réforme des retraites et celle de l’assurance chômage ? »
    {
        "dateSeance_day": "2024-04-05",
        "numSeanceJour": 0,
        "valeur_ptsodj": 1,
    },  #  « La place dans la société et dans le droit des familles monoparentales »
    {
        "dateSeance_day": "2024-04-03",
        "numSeanceJour": 2,
        "valeur_ptsodj": 1,
    },  # Bilan des réformes de l’assurance chômage depuis 2017
    {
        "dateSeance_day": "2024-04-03",
        "numSeanceJour": 1,
        "valeur_ptsodj": 3,
    },  # Conditions d’accueil des enfants placés à l’aide sociale à l’enfance
    {
        "dateSeance_day": "2024-04-03",
        "numSeanceJour": 1,
        "valeur_ptsodj": 2,
    },  # Défaillances de l’aide sociale à l’enfance
    {
        "dateSeance_day": "2024-02-28",
        "numSeanceJour": 2,
        "valeur_ptsodj": 1,
    },  # Conséquences de la loi « immigration » sur les enfants étrangers placés à l’aide sociale à l’enfance
    {
        "dateSeance_day": "2024-02-28",
        "numSeanceJour": 1,
        "valeur_ptsodj": 1,
    },  # Défaillances de l’aide sociale à l’enfance
    {
        "dateSeance_day": "2020-01-09",
        "numSeanceJour": 0,
        "valeur_ptsodj": 2,
    },  # projet hercule
    {
        "dateSeance_day": "2023-01-09",
        "numSeanceJour": 1,
        "valeur_ptsodj": 2,
    },  # Réforme de la voie professionnelle
    {
        "dateSeance_day": "2022-01-31",
        "numSeanceJour": 1,
        "valeur_ptsodj": 2,
    },  # Évaluation du plan gouvernemental L’État plus fort en Seine-Saint-Denis
    {
        "dateSeance_day": "2022-01-06",
        "numSeanceJour": 1,
        "valeur_ptsodj": 2,
    },  # Sahara occidental
    {
        "dateSeance_day": "2022-01-06",
        "numSeanceJour": 1,
        "valeur_ptsodj": 1,
    },  # Légalisation du cannabis : évolutions européennes,
    {
        "dateSeance_day": "2021-05-03",
        "numSeanceJour": 1,
        "valeur_ptsodj": 3,
    },  # Bilan de la loi ÉGALIM sur la rémunération des agriculteurs
    {
        "dateSeance_day": "2020-02-06",
        "numSeanceJour": 0,
        "valeur_ptsodj": 2,
    },  # D ébat sur les allégements de la fiscalité
]

# Initialiser une liste pour stocker les DataFrames filtrés
dfs_filtres = []

# Appliquer les filtres
for filtre in filtres:
    date = filtre["dateSeance_day"]
    num_seance = filtre["numSeanceJour"]
    valeur_pt = filtre[
        "valeur_ptsodj"
    ]  # TODO: LM : doute ici avec structure des fichiers

    # Filtrer par date et numéro de séance
    df_temp = df[(df["dateSeance_day"] == date) & (df["numSeanceJour"] == num_seance)]

    # # Si une valeur de point est spécifiée, filtrer aussi par cette valeur
    # if valeur_pt is not None:
    #     df_temp = df_temp[df_temp["valeur_ptsodj"] == valeur_pt]

    # Ajouter le DataFrame filtré à la liste
    dfs_filtres.append(df_temp)

# Concaténer tous les DataFrames filtrés
df_filtres_sans_ptsodj = pd.concat(dfs_filtres, ignore_index=True)

In [30]:
df_filtres.shape

(2998, 31)

In [31]:
df_filtres_sans_ptsodj.shape

(7173, 31)

In [32]:
len(df_filtres) / len(df) * 100

0.26581831787898613

In [33]:
len(df_filtres_sans_ptsodj) / len(df) * 100

0.6359955951120639

In [34]:
# Vérification des différences entre df_filtres et df_filtres_sans_ptsodj

# 1) Tailles
print("df_filtres.shape =", df_filtres.shape)
print("df_filtres_sans_ptsodj.shape =", df_filtres_sans_ptsodj.shape)

# 2) Colonnes communes (au cas où l'ordre/nb de colonnes diffère)
cols = sorted(set(df_filtres.columns).intersection(df_filtres_sans_ptsodj.columns))

# 3) Lignes uniquement dans l'un ou dans l'autre (comparaison exacte sur les colonnes communes)
a = df_filtres[cols].copy()
b = df_filtres_sans_ptsodj[cols].copy()

diff = a.merge(b, on=cols, how="outer", indicator=True)

only_sans_ptsodj = diff[diff["_merge"] == "right_only"].drop(columns="_merge")

print("Lignes uniquement dans df_filtres_sans_ptsodj :", len(only_sans_ptsodj))

# 4) Aperçu des différences
display(only_sans_ptsodj.head(5))

df_filtres.shape = (2998, 31)
df_filtres_sans_ptsodj.shape = (7173, 31)
Lignes uniquement dans df_filtres_sans_ptsodj : 3181


,SeanceRef,SessionRef,code_grammaire,code_parole,code_style,dateSeance,dateSeanceJour,dateSeance_day,dateseance_ts,id_acteur,...,point_type,presidentSeance,qualite_orateur,roledebat,session,stime,texte,typeAssemblee,uid,valeur_ptsodj
0,RUANR5L15S2021IDS23990,SCR5A2021O1,FIN_SEAN_2_1,NaN,NORMAL,20210322160000000,lundi 22 mars 2021,2021-03-22,2021-03-22 16:00:00,PA605991,...,FIN_SEAN_1_2,Présidence de Mme Annie Genevard,NaN,president,session ordinaire 2020-2021,NaN,"Prochaine séance, ce soir, à vingt et une heur...",AN,CRSANR5L15S2021O1N207,3
1,RUANR5L15S2021IDS23990,SCR5A2021O1,FIN_SEAN_2_4,NaN,Info Italiques,20210322160000000,lundi 22 mars 2021,2021-03-22,2021-03-22 16:00:00,NaN,...,FIN_SEAN_1_2,Présidence de Mme Annie Genevard,NaN,NaN,session ordinaire 2020-2021,NaN,(La séance est levée à vingt heures.),AN,CRSANR5L15S2021O1N207,3
2,RUANR5L15S2021IDS23990,SCR5A2021O1,FIN_SEAN_2_5,NaN,Signature droite,20210322160000000,lundi 22 mars 2021,2021-03-22,2021-03-22 16:00:00,NaN,...,FIN_SEAN_1_2,Présidence de Mme Annie Genevard,NaN,NaN,session ordinaire 2020-2021,NaN,Le directeur des comptes rendus\n \...,AN,CRSANR5L15S2021O1N207,3
16,RUANR5L15S2021IDS23990,SCR5A2021O1,INTERRUPTION_1_10,NaN,NORMAL,20210322160000000,lundi 22 mars 2021,2021-03-22,2021-03-22 16:00:00,PA719728,...,TITRE_TEXTE_DISCUSSION,Présidence de Mme Annie Genevard,NaN,NaN,session ordinaire 2020-2021,NaN,Oh !,AN,CRSANR5L15S2021O1N207,1
20,RUANR5L15S2021IDS23990,SCR5A2021O1,INTERRUPTION_1_10,NaN,NORMAL,20210322160000000,lundi 22 mars 2021,2021-03-22,2021-03-22 16:00:00,PA721062,...,TITRE_TEXTE_DISCUSSION,Présidence de Mme Annie Genevard,NaN,NaN,session ordinaire 2020-2021,NaN,Bonne question !,AN,CRSANR5L15S2021O1N207,1


In [35]:
only_sans_ptsodj

,SeanceRef,SessionRef,code_grammaire,code_parole,code_style,dateSeance,dateSeanceJour,dateSeance_day,dateseance_ts,id_acteur,...,point_type,presidentSeance,qualite_orateur,roledebat,session,stime,texte,typeAssemblee,uid,valeur_ptsodj
0,RUANR5L15S2021IDS23990,SCR5A2021O1,FIN_SEAN_2_1,NaN,NORMAL,20210322160000000,lundi 22 mars 2021,2021-03-22,2021-03-22 16:00:00,PA605991,...,FIN_SEAN_1_2,Présidence de Mme Annie Genevard,NaN,president,session ordinaire 2020-2021,NaN,"Prochaine séance, ce soir, à vingt et une heur...",AN,CRSANR5L15S2021O1N207,3
1,RUANR5L15S2021IDS23990,SCR5A2021O1,FIN_SEAN_2_4,NaN,Info Italiques,20210322160000000,lundi 22 mars 2021,2021-03-22,2021-03-22 16:00:00,NaN,...,FIN_SEAN_1_2,Présidence de Mme Annie Genevard,NaN,NaN,session ordinaire 2020-2021,NaN,(La séance est levée à vingt heures.),AN,CRSANR5L15S2021O1N207,3
2,RUANR5L15S2021IDS23990,SCR5A2021O1,FIN_SEAN_2_5,NaN,Signature droite,20210322160000000,lundi 22 mars 2021,2021-03-22,2021-03-22 16:00:00,NaN,...,FIN_SEAN_1_2,Présidence de Mme Annie Genevard,NaN,NaN,session ordinaire 2020-2021,NaN,Le directeur des comptes rendus\n \...,AN,CRSANR5L15S2021O1N207,3
16,RUANR5L15S2021IDS23990,SCR5A2021O1,INTERRUPTION_1_10,NaN,NORMAL,20210322160000000,lundi 22 mars 2021,2021-03-22,2021-03-22 16:00:00,PA719728,...,TITRE_TEXTE_DISCUSSION,Présidence de Mme Annie Genevard,NaN,NaN,session ordinaire 2020-2021,NaN,Oh !,AN,CRSANR5L15S2021O1N207,1
20,RUANR5L15S2021IDS23990,SCR5A2021O1,INTERRUPTION_1_10,NaN,NORMAL,20210322160000000,lundi 22 mars 2021,2021-03-22,2021-03-22 16:00:00,PA721062,...,TITRE_TEXTE_DISCUSSION,Présidence de Mme Annie Genevard,NaN,NaN,session ordinaire 2020-2021,NaN,Bonne question !,AN,CRSANR5L15S2021O1N207,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7450,NaN,NaN,SUSP_SEANCE_2_1,NaN,NORMAL,20200109090000000,jeudi 09 janvier 2020,2020-01-09,2020-01-09 09:00:00,PA721824,...,SUSP_SEANCE_1_1,Présidence de M. Hugues Renson,NaN,president,session ordinaire 2019-2020,NaN,La séance est suspendue.,AN,CRSANR5L15S2020O1N119,1
7452,NaN,NaN,SUSP_SEANCE_2_1,NaN,NORMAL,20200206090000000,jeudi 06 février 2020,2020-02-06,2020-02-06 09:00:00,PA1874,...,SUSP_SEANCE_1_1,Présidence de M. Marc Le Fur,NaN,president,session ordinaire 2019-2020,NaN,La séance est suspendue.,AN,CRSANR5L15S2020O1N137,1
7453,NaN,NaN,SUSP_SEANCE_2_2,NaN,Info Italiques,20200109090000000,jeudi 09 janvier 2020,2020-01-09,2020-01-09 09:00:00,NaN,...,SUSP_SEANCE_1_1,Présidence de M. Hugues Renson,NaN,NaN,session ordinaire 2019-2020,NaN,"(La séance, suspendue à dix heures cinquante, ...",AN,CRSANR5L15S2020O1N119,1
7455,NaN,NaN,SUSP_SEANCE_2_2,NaN,Info Italiques,20200206090000000,jeudi 06 février 2020,2020-02-06,2020-02-06 09:00:00,NaN,...,SUSP_SEANCE_1_1,Présidence de M. Marc Le Fur,NaN,NaN,session ordinaire 2019-2020,NaN,"(La séance, suspendue à dix heures quarante, e...",AN,CRSANR5L15S2020O1N137,1
